# 06b · Modelado: Seq2Seq BiLSTM con atención multi-cabeza + Kalman (Propuesta B, ensamble)

| | |
|---|---|
| **Fase CRISP-DM** | 4 — Modelado |
| **Objetivo** | Entrenar el modelo de la Propuesta B como 5.º modelo de la comparación: encoder-decoder recurrente con atención y generación **paso a paso**, sobre la señal filtrada con Kalman, como ensamble de varias semillas. |
| **Entradas** | `data/gold/series_filtradas.parquet` (voltaje real, calidad, covariables; solo train), `data/gold/series_kalman.parquet`, `data/gold/escaladores_kalman.json` |
| **Salidas** | `models/seq2seq_att_kalman_s{semilla}.pt`, `models/seq2seq_att_kalman.json` (manifiesto del ensamble), `reports/modelado/seq2seq_*` |

**Arquitectura**
- **Encoder:** BiLSTM sobre la ventana de 30 días `[B, 30, 4]` (mismos canales que el modelo A, con la señal de Kalman en lugar de VMD).
- **Decoder:** `LSTMCell` inicializada con el estado final del encoder, la identidad de la celda y el electrolizador. En cada paso, atención multi-cabeza (4 cabezas) sobre los 30 estados del encoder.
- **Generación autorregresiva:** `ŷ_{t0+k}` se calcula a partir de `ŷ_{t0+k−1}`, hasta 180 pasos. Se entrena **sin teacher forcing** (el decoder siempre se alimenta con su propia predicción, igual que en inferencia), porque muchos objetivos están enmascarados.
- **Ensamble:** una red por semilla (`ENSEMBLE_SEEDS`); el pronóstico final es el promedio.

**Mismo protocolo que el modelo A:** bloque de early stopping, máscara de saltos de nivel, MSE enmascarado, gradient clipping y reentrenamiento final con todo train. Por el costo del decoder de 180 pasos, los orígenes de train se submuestrean cada 10 días (5 en el modelo A).

In [ ]:
# Configuración: localiza la raíz del proyecto (local o Colab) y agrega src/ al path
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/Poc-TP1")
except ImportError:
    ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "voltcast").is_dir())
sys.path.insert(0, str(ROOT / "src"))

In [ ]:
import json
import time
import warnings

warnings.filterwarnings("ignore", category=UserWarning)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch

from voltcast import cleaning as cl
from voltcast import config, io
from voltcast import dataset as ds
from voltcast import model as mdl

config.ensure_dirs()
FIG_DIR = config.REPORTS_DIR / "modelado"
FIG_DIR.mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid", context="notebook")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CFG = config.SEQ2SEQ
print(f"Dispositivo: {DEVICE} | semillas: {config.ENSEMBLE_SEEDS}")
print("Configuración:", CFG)


def savefig(fig, name):
    path = FIG_DIR / f"{name}.png"
    fig.savefig(path, dpi=130, bbox_inches="tight")
    print(f"Figura guardada: {path.relative_to(config.BASE_DIR)}")

## 1. Datos de train con la señal de Kalman

In [ ]:
gold = io.read_layer("gold", "series_filtradas.parquet")
kal = io.read_layer("gold", "series_kalman.parquet")
esc = io.read_json("gold", "escaladores_kalman.json")
SIGMA = esc["sigma_delta"]
train = gold[gold["split"] == "train"].drop(columns="v_filtrada")
train["celda"] = train["celda"].astype(str)
kal["celda"] = kal["celda"].astype(str)
train = train.merge(kal[["fecha", "celda", "v_kalman"]], on=["fecha", "celda"], how="left", validate="one_to_one")
train = train.rename(columns={"v_kalman": "v_filtrada"})  # la clase de tensores usa la columna 'v_filtrada'
CELLS = sorted(train["celda"].unique())

data = ds.SeriesTensors(train, CELLS, SIGMA, DEVICE)
T_ = len(data.dates)
CUT = T_ - config.ES_BLOCK_DAYS
TR_RANGE = (config.LOOKBACK_DAYS - 1, CUT - 1)
VAL_PAIRS = data.origins(CUT, T_ - 2, stride=7)
V_tr = ds.pivot(train, "voltaje", CELLS)
Q_tr = ds.pivot(train, "calidad", CELLS).apply(lambda s: pd.Categorical(s, categories=cl.CALIDAD).codes)
data.set_level_shifts(cl.level_shifts(V_tr, Q_tr))
print(f"σ_Δ (Kalman) = {SIGMA:.4f} V | orígenes usables: {int(data.ok.sum()):,} | "
      f"bloque interno: {len(VAL_PAIRS[0]):,} pares desde {data.dates[CUT].date()}")

## 2. Entrenamiento del ensamble

In [ ]:
MV = {}  # modelos del early stopping (para el bloque interno)
res = {}
for seed in config.ENSEMBLE_SEEDS:
    print(f"\n=== Semilla {seed}: early stopping ===")
    torch.manual_seed(seed)
    m_es = mdl.build_model(len(CELLS), len(ds.FEATURES), CFG).to(DEVICE)
    t = time.time()
    r_es = mdl.fit(m_es, data, TR_RANGE, CUT - 1, CFG, val_pairs=VAL_PAIRS, val_target_limit=T_ - 1,
                   seed=seed, mask_jumps=config.MASK_POST_JUMP, log=print)
    t_es = time.time() - t
    print(f"→ mejor época {r_es['mejor_epoca']} ({t_es / 60:.1f} min, {mdl.count_params(m_es):,} parámetros)")
    print(f"=== Semilla {seed}: reentrenamiento con todo train ({r_es['mejor_epoca']} épocas) ===")
    torch.manual_seed(seed)
    final = mdl.build_model(len(CELLS), len(ds.FEATURES), CFG).to(DEVICE)
    t = time.time()
    r_fin = mdl.fit(final, data, (config.LOOKBACK_DAYS - 1, T_ - 2), T_ - 1, CFG, epochs=r_es["mejor_epoca"],
                    seed=seed, mask_jumps=config.MASK_POST_JUMP, log=None)
    t_fin = time.time() - t
    print(f"→ {t_fin / 60:.1f} min")
    MV[seed] = m_es
    res[seed] = {"final": final, "r_es": r_es, "r_fin": r_fin, "minutos": (t_es + t_fin) / 60}

## 3. Resultados en el bloque interno de train

In [ ]:
def fila(nombre, rm):
    return {"modelo": nombre, **{f"RMSE h{h} (V)": float(rm[h - 1]) * SIGMA for h in (1, 30, 60, 180)},
            "RMSE medio 1–180 (V)": float(rm.mean()) * SIGMA}


filas, curvas = [], {}
for seed, m in MV.items():
    rm, rp = mdl.rmse_by_horizon(m, data, *VAL_PAIRS, T_ - 1, batch_size=2048)
    filas.append(fila(f"Seq2Seq semilla {seed}", rm))
ens_es = mdl.Ensemble(list(MV.values())).eval()
rm, rp = mdl.rmse_by_horizon(ens_es, data, *VAL_PAIRS, T_ - 1, batch_size=1024)
curvas["Seq2Seq-Att + Kalman (ensamble)"] = rm.cpu().numpy() * SIGMA
curvas["persistencia (ancla Kalman)"] = rp.cpu().numpy() * SIGMA
filas += [fila("Seq2Seq-Att + Kalman (ensamble)", rm), fila("persistencia (ancla Kalman)", rp)]
bloque = pd.DataFrame(filas).set_index("modelo")
bloque["mejor época"] = [res[s]["r_es"]["mejor_epoca"] for s in MV] + [None, None]
bloque["tiempo (min)"] = [round(res[s]["minutos"], 1) for s in MV] + [None, None]

# Referencia: modelo A (notebook 06) en el mismo bloque
ref = pd.read_csv(FIG_DIR / "metricas_bloque_interno.csv", index_col=0)
bloque = pd.concat([bloque, ref.loc[["Bi-NOA-LSTM", "Bi-LSTM estándar"]]])
bloque.to_csv(FIG_DIR / "seq2seq_metricas_bloque_interno.csv")
bloque.round(4)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for seed in MV:
    h = pd.DataFrame(res[seed]["r_es"]["historia"])
    l, = axes[0].plot(h["epoca"], h["train_mse"], label=f"semilla {seed} · train")
    axes[0].plot(h["epoca"], h["val_mse"], ls="--", color=l.get_color(), label=f"semilla {seed} · bloque interno")
axes[0].set(yscale="log", xlabel="época", ylabel="MSE (normalizado)", title="Seq2Seq-Att + Kalman: curvas de entrenamiento")
for n, c in curvas.items():
    axes[1].plot(range(1, config.HORIZON_DAYS + 1), c, ls="--" if "persistencia" in n else "-",
                 color="k" if "persistencia" in n else "tab:green", label=n)
axes[1].set(xlabel="horizonte (días)", ylabel="RMSE (V)", title="Error vs. horizonte en el bloque interno de train")
axes[0].legend(fontsize=7); axes[1].legend(fontsize=8)
fig.tight_layout(); savefig(fig, "03_seq2seq_curvas")

**Lectura.**
- **Bloque interno:** el ensamble supera a la persistencia a 60 días (0,0215 frente a 0,0223 V) y a 180 días (0,0318 frente a 0,0347 V). Queda cerca de la Bi-NOA-LSTM, aunque algo peor a 180 días (0,0297 V).
- **Estabilidad por semilla:**
  - Las semillas 42 y 43 entrenan de forma estable (mejores épocas 20 y 17).
  - **La semilla 44 diverge en la época 5**: el MSE de train salta de ≈ 0,08 a > 1 y no se recupera. El early stopping conserva sus pesos de la época 3, por lo que aporta al ensamble un modelo poco entrenado (el peor de los tres: 0,0333 V a 180 días).
  - Es una inestabilidad conocida de los decoders recurrentes autorregresivos: un error grande se realimenta en los pasos siguientes. El promedio del ensamble la amortigua. Se mitigaría con una tasa de aprendizaje menor o con un calentamiento (*warm-up*) de la tasa.
- **Error vs. horizonte:** en este bloque no se observa deriva por recursión. El RMSE crece de forma parecida al del modelo de salida directa, porque el decoder aprendió trayectorias suaves.
- **Costo:** ~73 min para el ensamble (3 semillas), frente a ~5 min de la Bi-NOA-LSTM. El decoder de 180 pasos y el uso de la memoria de la GPU lo hacen mucho más caro de entrenar, pese a submuestrear los orígenes cada 10 días en lugar de 5.

## 4. Guardado del ensamble

In [ ]:
kalman_params = json.loads((config.MODELS_DIR / "kalman_params.json").read_text(encoding="utf-8"))
archivos, historial = [], []
for seed, r in res.items():
    meta = {
        "nombre": "Seq2Seq-Att + Kalman", "arch": "seq2seq", "seed": seed, "cells": CELLS, "features": ds.FEATURES,
        "train_cfg": CFG, "lookback": config.LOOKBACK_DAYS, "horizon": config.HORIZON_DAYS,
        "anchor_days": config.ANCHOR_DAYS, "escaladores": esc, "filtro": "kalman", "kalman_params": kalman_params,
        "mask_post_jump": config.MASK_POST_JUMP, "refit_full_train": True, "mejor_epoca": r["r_es"]["mejor_epoca"],
        "entrenado_hasta": str(data.dates[-1].date()), "es_block_desde": str(data.dates[CUT].date()),
        "fecha_entrenamiento": pd.Timestamp.now().isoformat(timespec="seconds"),
    }
    path = config.MODELS_DIR / f"seq2seq_att_kalman_s{seed}.pt"
    mdl.save_checkpoint(path, r["final"], meta)
    archivos.append(path.name)
    print(f"✔ {path.relative_to(config.BASE_DIR)} ({path.stat().st_size / 1e6:.2f} MB)")
    historial += [{"semilla": seed, "fase": "early_stopping", **h} for h in r["r_es"]["historia"]]
    historial += [{"semilla": seed, "fase": "reentrenamiento", **h} for h in r["r_fin"]["historia"]]
manifiesto = {"nombre": "Seq2Seq-Att + Kalman (ensamble)", "combinacion": "promedio", "checkpoints": archivos,
              "semillas": config.ENSEMBLE_SEEDS, "escaladores": "data/gold/escaladores_kalman.json",
              "senal_entrada": "data/gold/series_kalman.parquet"}
(config.MODELS_DIR / "seq2seq_att_kalman.json").write_text(json.dumps(manifiesto, indent=2, ensure_ascii=False), encoding="utf-8")
pd.DataFrame(historial).to_csv(FIG_DIR / "seq2seq_historial_entrenamiento.csv", index=False)
print("✔ models/seq2seq_att_kalman.json y reports/modelado/seq2seq_historial_entrenamiento.csv")

## 5. Verificaciones

In [ ]:
x, y, mask, cell, el = data.batch(VAL_PAIRS[0][:256], VAL_PAIRS[1][:256], T_ - 1)
checks = {}
for seed, archivo in zip(res, archivos):
    m, ck = mdl.load_checkpoint(config.MODELS_DIR / archivo, DEVICE)
    with torch.no_grad():
        p1, p2 = m(x, cell, el), res[seed]["final"].eval()(x, cell, el)
    checks[f"semilla {seed}: el checkpoint reproduce las predicciones"] = torch.allclose(p1, p2, atol=1e-5)
    checks[f"semilla {seed}: salida [B, {config.HORIZON_DAYS}] finita"] = p1.shape[1] == config.HORIZON_DAYS and bool(torch.isfinite(p1).all())
checks["solo se usaron fechas de train"] = data.dates[-1] < gold.loc[gold["split"] == "val", "fecha"].min()
for k, v in checks.items():
    print(("✔" if v else "✘"), k)
assert all(checks.values())

## Conclusión

- Se entrenó la **Propuesta B** como ensamble de 3 semillas: encoder BiLSTM, decoder LSTM con atención multi-cabeza, generación paso a paso y señal de Kalman. Queda guardada en `models/seq2seq_att_kalman_s{42,43,44}.pt` + `models/seq2seq_att_kalman.json`.
- En el bloque interno supera a la persistencia y queda cerca de la Bi-NOA-LSTM, a un costo de entrenamiento ~15 veces mayor.
- Una de las tres semillas fue inestable. Es un riesgo propio de esta arquitectura y conviene tenerlo en cuenta si se elige para producción.

**Siguiente:** notebook 07 — la comparación de los 5 modelos en validación.